# DB → LLM SQL → FHIR 단계별 테스트

이 노트북은 현재 프로젝트를 다음 순서로 점검합니다.

1. `.env`와 연결 설정
2. Demo Patient NDJSON에서 환자 key 추출
3. PostgreSQL 사용자 스키마 자동 탐색
4. SQL/FHIR Vector DB 및 RAG 검색
5. LLM SQL 생성·리소스별 DB 행 조회·Self-Correction
6. 단일 환자 LLM FHIR 생성
7. FHIR Bundle 검증
8. 선택적 demo 환자 10명 배치 실행

비밀번호는 출력하지 않습니다. LLM 호출, Vector 재구축, 원천 DB 행 출력, 외부 Validator 및 배치 실행 여부는 아래 `RUN_*` 플래그의 현재 값을 확인해 제어하세요.

In [1]:
from pathlib import Path
import json
import sys
from dataclasses import replace
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "main.py").is_file():
    candidates = [p for p in PROJECT_ROOT.parents if (p / "main.py").is_file()]
    if not candidates:
        raise RuntimeError("main.py가 있는 프로젝트 루트에서 노트북을 실행하세요.")
    PROJECT_ROOT = candidates[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 통합 테스트 모드입니다. 실행하지 않을 단계는 False로 바꿉니다.
RUN_VECTOR_REBUILD = False
RUN_RAG_SEARCH = True
RUN_LLM_SQL = True
RUN_RESOURCE_DB_ROWS = True
RUN_SINGLE_PATIENT_PIPELINE = False
RUN_EXTERNAL_VALIDATOR = True
RUN_BATCH_10 = False

NOTEBOOK_OUTPUT = PROJECT_ROOT / "fhir_export_notebook_test"
DB_ROW_DISPLAY_LIMIT = 20
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /home/jhkim/fhir_agent


## 1. `.env` 및 실행 설정 테스트

DB URL은 출력하지 않고 설정 로딩, 파일 경로, 모델과 Vector 테이블 이름만 확인합니다.

In [2]:
from fhir_agent_pipeline.config import load_settings

settings = load_settings()
config_summary = {
    "source_profile": str(settings.source_profile_path),
    "source_data_path": str(settings.source_data_path),
    "fhir_schema_path": str(settings.fhir_schema_path),
    "ollama_model": settings.ollama_model,
    "vectors_enabled": settings.vectors_enabled,
    "sql_vector_table": settings.sql_vector_table,
    "fhir_vector_table": settings.fhir_vector_table,
    "sql_max_rows": settings.sql_max_rows,
    "max_observations_per_encounter": settings.max_observations_per_encounter,
    "llm_fhir_batch_size": settings.llm_fhir_batch_size,
    "output_resources": settings.fhir_output_resources,
    "validator_configured": settings.fhir_validator_jar is not None,
}
assert settings.source_profile_path.is_file()
assert settings.fhir_schema_path.is_file()
display(config_summary)


{'source_profile': '/home/jhkim/fhir_agent/profiles/mimic_iv/profile.json',
 'source_data_path': '/home/mdhi/physionet.org/files/mimic-iv-fhir-demo',
 'fhir_schema_path': '/home/jhkim/fhir_agent/db/fhir.schema.json',
 'ollama_model': 'gemma4:e4b',
 'vectors_enabled': True,
 'sql_vector_table': 'source_sql_schema_vectors',
 'fhir_vector_table': 'fhir_schema_vectors',
 'sql_max_rows': 10000,
 'max_observations_per_encounter': 2000,
 'llm_fhir_batch_size': 2,
 'output_resources': ['Patient',
  'Encounter',
  'Condition',
  'Procedure',
  'Observation'],
 'validator_configured': True}

### 1-1. FHIR Schema 기반 데이터형 정규화 테스트

병원별 매핑 없이 로컬 FHIR JSON Schema를 읽어 scalar 원천 값을 FHIR complex datatype 배열 형태로 보정하는지 확인합니다.

In [3]:
from fhir_agent_pipeline.fhir_schema import get_fhir_schema

fhir_schema = get_fhir_schema(settings.fhir_schema_path)
interpretation_schema = fhir_schema.property_schema("Observation", "interpretation")
assert interpretation_schema is not None
normalized_interpretation = fhir_schema.coerce_source_value(
    interpretation_schema,
    "abnormal",
)
assert normalized_interpretation == [{"text": "abnormal"}]
display({
    "schema": interpretation_schema,
    "source": "abnormal",
    "normalized": normalized_interpretation,
})

{'schema': {'description': 'A categorical assessment of an observation value.  For example, high, low, normal.',
  'items': {'$ref': '#/definitions/CodeableConcept'},
  'type': 'array'},
 'source': 'abnormal',
 'normalized': [{'text': 'abnormal'}]}

## 2. Demo cohort 환자 key 추출 테스트

LLM은 환자 ID를 추론하지 않습니다. Source profile이 지정한 Patient NDJSON identifier에서 key를 읽습니다.

In [4]:
from fhir_agent_pipeline.export_service import (
    iter_cohort_patient_ids,
    resolve_cohort_patient_file,
)

cohort_file = resolve_cohort_patient_file(settings)
patient_keys = list(iter_cohort_patient_ids(settings, limit=10))
assert cohort_file.is_file()
assert len(patient_keys) == 10
assert len(set(patient_keys)) == 10
PATIENT_KEY = patient_keys[0]
display({"cohort_file": str(cohort_file), "patient_keys": patient_keys})


{'cohort_file': '/home/mdhi/physionet.org/files/mimic-iv-fhir-demo/2.1.0/fhir/MimicPatient.ndjson.gz',
 'patient_keys': [10007795,
  10007928,
  10009628,
  10011398,
  10004457,
  10003046,
  10014078,
  10003400,
  10020187,
  10038999]}

## 3. PostgreSQL 스키마 자동 탐색 테스트

`public`, `information_schema`, `pg_*`, `*derived*`를 제외한 사용자 스키마의 테이블·컬럼·PK·FK를 DB metadata에서 읽습니다.

In [5]:
from fhir_agent_pipeline.db import build_engine, SourceSchemaCatalog

engine = build_engine(settings)
try:
    catalog = SourceSchemaCatalog.load(settings, engine)
    assert catalog.tables
    assert all(t.schema.lower() not in {"public", "information_schema"} for t in catalog.tables)
    assert all(not t.schema.lower().startswith("pg_") for t in catalog.tables)
    schema_summary = {
        "schemas": sorted(catalog.schemas),
        "table_count": len(catalog.tables),
        "sample_tables": [t.qualified_name for t in catalog.tables[:10]],
    }
    display(schema_summary)
finally:
    engine.dispose()


{'schemas': ['mimiciv_derived', 'mimiciv_hosp', 'mimiciv_icu'],
 'table_count': 59,
 'sample_tables': ['mimiciv_derived.antibiotic',
  'mimiciv_derived.bg',
  'mimiciv_derived.chemistry',
  'mimiciv_derived.complete_blood_count',
  'mimiciv_derived.crrt',
  'mimiciv_derived.dobutamine',
  'mimiciv_derived.dopamine',
  'mimiciv_derived.enzyme',
  'mimiciv_derived.epinephrine',
  'mimiciv_derived.gcs']}

## 4. Vector DB 및 RAG 검색 테스트

기본 실행은 기존 Vector 테이블을 검사합니다. `RUN_VECTOR_REBUILD=True`이면 현재 DB metadata와 FHIR JSON Schema로 Vector DB를 갱신합니다.

In [6]:
from fhir_agent_pipeline.vector_store import (
    PgVectorKnowledgeStore,
    build_knowledge_bases,
)

if RUN_VECTOR_REBUILD:
    rebuild_result = build_knowledge_bases(settings)
    display(rebuild_result)

store = PgVectorKnowledgeStore(settings)
try:
    vector_reports = [
        store.inspect_table(settings.sql_vector_table),
        store.inspect_table(settings.fhir_vector_table),
    ]
    for report in vector_reports:
        assert report["exists"] and report["compatible"] and report["rows"] > 0
    display(vector_reports)

    if RUN_RAG_SEARCH:
        sql_matches = store.search(
            settings.sql_vector_table,
            "patient demographics encounter identifier and admission time",
            k=5,
            profile_scoped=True,
        )
        fhir_matches = store.search(
            settings.fhir_vector_table,
            "FHIR R4 Patient required properties references datatypes",
            k=5,
            profile_scoped=False,
        )
        assert sql_matches and fhir_matches
        display({
            "sql_rag": [
                {"metadata": x["metadata"], "similarity": round(float(x["similarity"]), 4)}
                for x in sql_matches
            ],
            "fhir_rag": [
                {"metadata": x["metadata"], "similarity": round(float(x["similarity"]), 4)}
                for x in fhir_matches
            ],
        })
finally:
    store.engine.dispose()


[{'table': 'public.source_sql_schema_vectors',
  'exists': True,
  'columns': {'id': 'bigint',
   'content': 'text',
   'metadata': 'jsonb',
   'embedding': 'vector(768)',
   'content_hash': 'character varying(64)',
   'updated_at': 'timestamp with time zone'},
  'rows': 59,
  'compatible': True,
  'indexes': ['source_sql_schema_vectors_content_hash_key',
   'source_sql_schema_vectors_embedding_hnsw_39602121',
   'source_sql_schema_vectors_pkey'],
  'current_profile_rows': 59,
  'global_rows': 0},
 {'table': 'public.fhir_schema_vectors',
  'exists': True,
  'columns': {'id': 'bigint',
   'content': 'text',
   'metadata': 'jsonb',
   'embedding': 'vector(768)',
   'content_hash': 'character varying(64)',
   'updated_at': 'timestamp with time zone'},
  'rows': 11499,
  'compatible': True,
  'indexes': ['fhir_schema_vectors_content_hash_key',
   'fhir_schema_vectors_embedding_hnsw_39590601',
   'fhir_schema_vectors_pkey'],
  'current_profile_rows': 0,
  'global_rows': 11499}]

{'sql_rag': [{'metadata': {'kind': 'table_schema',
    'table': 'admissions',
    'schema': 'mimiciv_hosp',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',
    'profile_fingerprint': '8e114982adfd7428baba0eb08472d8264ca15b0f31aca36051321694ad80b178'},
   'similarity': 0.6297},
  {'metadata': {'kind': 'table_schema',
    'table': 'datetimeevents',
    'schema': 'mimiciv_icu',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',
    'profile_fingerprint': '8e114982adfd7428baba0eb08472d8264ca15b0f31aca36051321694ad80b178'},
   'similarity': 0.6005},
  {'metadata': {'kind': 'table_schema',
    'table': 'icustays',
    'schema': 'mimiciv_icu',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',
    'profile_fingerprint': '8e114982adfd7428baba0eb08472d8264ca15b0f31aca36051321694ad80b178'},
   'similarity': 0.587},
  {'metadata': {'kind': 'table_schema',
    'table': 'outputevents',
    'schema': 'mimiciv_icu',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',

## 5. LLM SQL → DB 조회 테스트

`RUN_LLM_SQL=True`로 바꾸면 선택 환자의 Patient 질의를 실행합니다. 출력은 SQL, 행 수, RAG 사용 여부와 교정 이력만 보여주며 전체 임상 행은 출력하지 않습니다.

In [7]:
sql_test_result = None

if RUN_LLM_SQL:
    from fhir_agent_pipeline.llm_query_service import EncounterQueryContext, LlmClinicalQueryService

    query_service = LlmClinicalQueryService(settings)
    try:
        queried = query_service.query(
            "Patient", PATIENT_KEY, EncounterQueryContext(), None
        )
        print(queried.trace.sql)
        sql_test_result = {
            "patient_key": PATIENT_KEY,
            "sql": queried.trace.sql,
            "row_count": len(queried.rows),
            "rag_requested": queried.trace.rag_requested,
            "rag_used": queried.trace.rag_used,
            "retrieved_documents": queried.trace.retrieved_document_count,
            "llm_calls": queried.trace.llm_calls_used,
            "attempts": len(queried.trace.attempts),
            "aliases": sorted(queried.rows[0]) if queried.rows else [],
        }
        assert queried.trace.rag_used
        assert len(queried.rows) == 1
        display(sql_test_result)
    finally:
        query_service.close()
else:
    print("건너뜀: RUN_LLM_SQL=True로 설정하면 실행됩니다.")


SELECT * FROM (
SELECT T1.subject_id AS _patient_key, T1.gender AS _gender, NULL AS _death_datetime
FROM mimiciv_hosp.patients AS T1 WHERE T1.subject_id = 10007795
) AS _safe_query
LIMIT 10000


{'patient_key': 10007795,
 'sql': 'SELECT * FROM (\nSELECT T1.subject_id AS _patient_key, T1.gender AS _gender, NULL AS _death_datetime\nFROM mimiciv_hosp.patients AS T1 WHERE T1.subject_id = 10007795\n) AS _safe_query\nLIMIT 10000',
 'row_count': 1,
 'rag_requested': True,
 'rag_used': True,
 'retrieved_documents': 30,
 'llm_calls': 1,
 'attempts': 1,
 'aliases': ['_death_datetime', '_gender', '_patient_key']}

### 5-1. 실제 Pipeline에 DB 행 기록기 연결

독립 SQL 조회를 다시 실행하지 않습니다. `RecordingQueryService`를 실제 `LlmSqlToFhirPipeline`에 주입하여 Pipeline이 FHIR 생성에 사용한 **동일한 SQL 결과 행**을 기록합니다. Encounter SQL에서 동적으로 추출한 relation lineage와 모든 임상 조회가 0건일 때 수행되는 semantic relationship retry도 함께 기록합니다.

다음 6번 셀에서 한 번의 실행으로 SQL·RAG·DB 행·FHIR 리소스·검증 결과를 함께 확인합니다. `RUN_RESOURCE_DB_ROWS=True`이면 원천 임상값이 표시되므로 승인된 환경에서만 실행하고 `DB_ROW_DISPLAY_LIMIT`으로 표시 행 수를 제한하세요.

In [8]:
from fhir_agent_pipeline.llm_query_service import LlmClinicalQueryService


class RecordingQueryService(LlmClinicalQueryService):
    """실제 Pipeline에 반환한 QueriedResourceRows를 그대로 기록합니다."""

    def __init__(self, settings):
        super().__init__(settings)
        self.recorded = {}
        self.history = []

    def query(
        self,
        resource_type,
        patient_key,
        encounter_context,
        encounter_override,
        **kwargs,
    ):
        queried = super().query(
            resource_type,
            patient_key,
            encounter_context,
            encounter_override,
            **kwargs,
        )
        self.recorded[resource_type] = queried
        self.history.append(queried)
        return queried


resource_db_results = {}
print("RecordingQueryService 준비 완료")

RecordingQueryService 준비 완료


## 6. 동일 DB 행을 사용하는 단일 환자 End-to-End 테스트

`RUN_SINGLE_PATIENT_PIPELINE=True` 또는 `RUN_RESOURCE_DB_ROWS=True`이면 Pipeline을 한 번만 실행합니다. 기록된 DB 행은 바로 그 실행의 FHIR 생성 입력이며, 리소스별 SQL·DB 행·생성 FHIR·평가 결과를 함께 확인할 수 있습니다. `encounter_context_relations`는 실행된 Encounter SQL에서 추출한 동적 lineage이고, `semantic_relationship_retry`는 병원별 매핑 없이 관계를 재검색한 trace인지 나타냅니다. 마지막 요약의 `encounter_scope_narrowing_rejections`와 `encounter_identity_rejections`로 일반 방문 범위를 축소하거나 Encounter 키를 중복시킨 SQL의 self-correction 여부도 확인합니다.

In [9]:
pipeline_result = None
recording_query_service = None
resource_db_results = {}

if RUN_SINGLE_PATIENT_PIPELINE or RUN_RESOURCE_DB_ROWS:
    from collections import Counter, defaultdict
    from fhir_agent_pipeline.llm_pipeline import LlmSqlToFhirPipeline

    try:
        import pandas as pd
    except ImportError:
        pd = None

    # 이 QueryService가 반환한 행을 Pipeline이 그대로 FHIR 생성에 사용합니다.
    recording_query_service = RecordingQueryService(settings)
    pipeline = LlmSqlToFhirPipeline(
        settings,
        query_service=recording_query_service,
    )
    pipeline_result = pipeline.run(PATIENT_KEY)

    generated_by_type = defaultdict(list)
    for resource in pipeline_result.generated_resources:
        generated_by_type[resource.get("resourceType")].append(resource)

    traces_by_type = {
        trace.resource_type: trace for trace in pipeline_result.query_traces
    }
    for resource_type in settings.fhir_output_resources:
        queried = recording_query_service.recorded.get(resource_type)
        trace = traces_by_type.get(resource_type)
        rows = queried.rows if queried is not None else []
        fhir_resources = generated_by_type.get(resource_type, [])
        resource_db_results[resource_type] = {
            "sql": trace.sql if trace else None,
            "row_count": len(rows),
            "columns": sorted(rows[0]) if rows else [],
            "rag_requested": trace.rag_requested if trace else False,
            "rag_used": trace.rag_used if trace else False,
            "retrieved_documents": trace.retrieved_document_count if trace else 0,
            "llm_calls": trace.llm_calls_used if trace else 0,
            "attempt_count": len(trace.attempts) if trace else 0,
            "failure_categories": list(trace.failure_categories) if trace else [],
            "encounter_context_relations": list(trace.encounter_context_relations) if trace else [],
            "semantic_relationship_retry": trace.semantic_relationship_retry if trace else False,
            "normalized_contract_aliases": list(trace.normalized_contract_aliases) if trace else [],
            "encounter_qualified_resource_keys": trace.encounter_qualified_resource_keys if trace else 0,
            "rows": rows,
            "fhir_resources": fhir_resources,
        }

        # Pipeline의 평가 분모와 기록한 실제 행 수가 동일해야 합니다.
        assert len(rows) == pipeline_result.source_row_counts.get(resource_type, 0)
        assert len(fhir_resources) == len(rows)

        if RUN_RESOURCE_DB_ROWS:
            print(f"\n===== {resource_type}: 동일 Pipeline 실행 =====")
            display({
                "patient_key": PATIENT_KEY,
                "row_count": len(rows),
                "fhir_resource_count": len(fhir_resources),
                "columns": resource_db_results[resource_type]["columns"],
                "rag_used": resource_db_results[resource_type]["rag_used"],
                "retrieved_documents": resource_db_results[resource_type]["retrieved_documents"],
                "llm_calls": resource_db_results[resource_type]["llm_calls"],
                "attempt_count": resource_db_results[resource_type]["attempt_count"],
                "failure_categories": resource_db_results[resource_type]["failure_categories"],
                "encounter_context_relations": resource_db_results[resource_type]["encounter_context_relations"],
                "semantic_relationship_retry": resource_db_results[resource_type]["semantic_relationship_retry"],
                "normalized_contract_aliases": resource_db_results[resource_type]["normalized_contract_aliases"],
                "encounter_qualified_resource_keys": resource_db_results[resource_type]["encounter_qualified_resource_keys"],
            })
            print("Generated SQL:\n", resource_db_results[resource_type]["sql"])
            print("DB rows used by FHIR generation:")
            rows_to_display = rows[:DB_ROW_DISPLAY_LIMIT]
            if pd is not None:
                display(pd.DataFrame(rows_to_display))
            else:
                display(rows_to_display)
            print("FHIR resources generated from the same rows:")
            display(fhir_resources[:DB_ROW_DISPLAY_LIMIT])

    resource_counts = Counter(
        resource.get("resourceType")
        for resource in pipeline_result.generated_resources
    )
    pipeline_summary = {
        "patient_key": PATIENT_KEY,
        "resource_counts": dict(resource_counts),
        "source_row_counts": pipeline_result.source_row_counts,
        "llm_calls": pipeline_result.llm_calls_used,
        "task_pass": pipeline_result.task_evaluation.get("task_quality_pass"),
        "validation_valid": pipeline_result.validation_report.valid,
        "sql_rag_used": [trace.rag_used for trace in pipeline_result.query_traces],
        "sql_trace_count": len(recording_query_service.history),
        "semantic_relationship_rechecks": pipeline_result.task_evaluation["sql"].get("semantic_relationship_rechecks", 0),
        "semantic_relationship_recoveries": pipeline_result.task_evaluation["sql"].get("semantic_relationship_recoveries", 0),
        "encounter_scope_narrowing_rejections": pipeline_result.task_evaluation["sql"].get("encounter_scope_narrowing_rejections", 0),
        "encounter_identity_rejections": pipeline_result.task_evaluation["sql"].get("encounter_identity_rejections", 0),
        "row_limit_reached_queries": pipeline_result.task_evaluation["sql"].get("row_limit_reached_queries", 0),
        "same_rows_verified": all(
            len(recording_query_service.recorded[name].rows)
            == pipeline_result.source_row_counts[name]
            == len(generated_by_type.get(name, []))
            for name in recording_query_service.recorded
        ),
    }
    assert pipeline_summary["task_pass"]
    assert pipeline_summary["validation_valid"]
    assert all(pipeline_summary["sql_rag_used"])
    assert pipeline_summary["same_rows_verified"]
    display(pipeline_summary)
else:
    print(
        "건너뜀: RUN_SINGLE_PATIENT_PIPELINE 또는 "
        "RUN_RESOURCE_DB_ROWS를 True로 설정하세요."
    )


===== Patient: 동일 Pipeline 실행 =====


{'patient_key': 10007795,
 'row_count': 1,
 'fhir_resource_count': 1,
 'columns': ['_death_datetime', '_gender', '_patient_key'],
 'rag_used': True,
 'retrieved_documents': 30,
 'llm_calls': 1,
 'attempt_count': 1,
 'failure_categories': [],
 'encounter_context_relations': [],
 'semantic_relationship_retry': False,
 'normalized_contract_aliases': [],
 'encounter_qualified_resource_keys': 0}

Generated SQL:
 SELECT * FROM (
SELECT T1.subject_id AS _patient_key, T1.gender AS _gender, NULL AS _death_datetime
FROM mimiciv_hosp.patients AS T1 WHERE T1.subject_id = 10007795
) AS _safe_query
LIMIT 10000
DB rows used by FHIR generation:


,_patient_key,_gender,_death_datetime
0,10007795,F,None


FHIR resources generated from the same rows:


[{'resourceType': 'Patient', 'id': '10007795', 'gender': 'female'}]

... (21 outputs left out)

""


FHIR resources generated from the same rows:


[]

{'patient_key': 10007795,
 'resource_counts': {'Patient': 1,
  'Encounter': 5,
  'Condition': 73,
  'Procedure': 1},
 'source_row_counts': {'Patient': 1,
  'Encounter': 5,
  'Condition': 73,
  'Procedure': 1,
  'Observation': 0},
 'llm_calls': 50,
 'task_pass': True,
 'validation_valid': True,
 'sql_rag_used': [True, True, True, True, True],
 'sql_trace_count': 5,
 'semantic_relationship_rechecks': 0,
 'semantic_relationship_recoveries': 0,
 'encounter_scope_narrowing_rejections': 0,
 'encounter_identity_rejections': 0,
 'row_limit_reached_queries': 0,
 'same_rows_verified': True}

## 7. FHIR Bundle 검증 테스트

직전 셀에서 생성한 Bundle이 있으면 사용하고, 없으면 보존된 demo 결과의 첫 Bundle을 사용합니다. 로컬 JSON Schema·구조·참조 검증은 항상 실행하며 외부 HL7 Validator는 플래그로 선택합니다.

In [10]:
from fhir_agent_pipeline.validation import validate_bundle_detailed

if pipeline_result is not None:
    bundle = pipeline_result.fhir_payload
    bundle_source = "현재 노트북에서 생성"
else:
    candidates = sorted((PROJECT_ROOT / "fhir_export_demo_generic_10").glob("patient-*.bundle.json"))
    if not candidates:
        raise FileNotFoundError("검증할 기존 Bundle이 없습니다. 단일 환자 파이프라인을 실행하세요.")
    bundle = json.loads(candidates[0].read_text(encoding="utf-8"))
    bundle_source = str(candidates[0])

local_settings = replace(settings, fhir_validator_jar=None)
local_report = validate_bundle_detailed(bundle, local_settings)
assert local_report.valid
display({
    "bundle_source": bundle_source,
    "local_valid": local_report.valid,
    "resource_count": local_report.metrics.get("resource_count"),
    "reference_resolution": local_report.metrics.get("reference_resolution_ratio"),
    "schema_valid_ratio": local_report.metrics.get("schema_valid_resource_ratio"),
})

if RUN_EXTERNAL_VALIDATOR:
    if settings.fhir_validator_jar is None:
        raise RuntimeError("FHIR_VALIDATOR_JAR가 설정되지 않았습니다.")
    external_report = validate_bundle_detailed(bundle, settings)
    assert external_report.external_validator_used
    display({
        "external_valid": external_report.valid,
        "issues": external_report.validator_issue_counts,
        "warning_count": len(external_report.warnings),
    })
else:
    print("외부 Validator 건너뜀: RUN_EXTERNAL_VALIDATOR=True로 설정하면 실행됩니다.")


{'bundle_source': '현재 노트북에서 생성',
 'local_valid': True,
 'resource_count': 80,
 'reference_resolution': 1.0,
 'schema_valid_ratio': 1.0}

{'external_valid': True,
 'issues': {'fatal': 0, 'error': 0, 'warning': 165, 'information': 0},
 'warning_count': 1}

## 8. 선택적 demo 10명 배치 테스트

`RUN_BATCH_10=True`이면 별도 `fhir_export_notebook_test/`에 10명을 저장하고 검증합니다. 이미 valid 및 task pass인 결과는 `resume=True`로 재사용합니다.

In [11]:
batch_summary = None
if RUN_BATCH_10:
    from fhir_agent_pipeline.export_service import ExportRequest, FhirBatchExportService

    request = ExportRequest(
        output_directory=NOTEBOOK_OUTPUT,
        limit=10,
        resume=True,
    )
    batch_summary = FhirBatchExportService(settings).run(request)
    display({
        "output_directory": batch_summary["output_directory"],
        "attempted": batch_summary["attempted_patients"],
        "valid": batch_summary["valid_patients"],
        "invalid": batch_summary["invalid_patients"],
        "failed": batch_summary["failed_patients"],
        "resources": batch_summary["resource_counts"],
        "performance": batch_summary["performance_evaluation"],
    })
    assert batch_summary["valid_patients"] == 10
    assert batch_summary["invalid_patients"] == 0
    assert batch_summary["failed_patients"] == 0
else:
    print("건너뜀: RUN_BATCH_10=True로 설정하면 실행됩니다.")


건너뜀: RUN_BATCH_10=True로 설정하면 실행됩니다.


## 판정 기준

- 설정·cohort·DB catalog·Vector 테이블 셀의 assertion이 모두 통과해야 합니다.
- SQL 테스트에서는 `rag_used=True`, Patient 1행과 필수 alias가 확인되어야 합니다.
- 리소스별 DB 행 테스트에서는 실제 Pipeline이 FHIR 생성에 사용한 동일 행과 생성 리소스 수가 일치하는지 확인합니다.
- 단일 환자 테스트에서는 모든 SQL query가 RAG를 사용하고 `task_pass=True`, `validation_valid=True`여야 합니다.
- 외부 Validator 테스트에서는 `external_validator_used=True`와 fatal/error 0건을 확인합니다.
- 배치 테스트에서는 valid 10, invalid 0, failed 0을 확인합니다.